# QymyzEmbed v0 — Less-is-More train + soup + eval (one Kaggle session)

End-to-end per arXiv:2603.22290 on KazParC pairs: generate → cap 10k → full fine-tune of
`intfloat/multilingual-e5-base` → 0.5/0.5 soup with base → evaluate on the pinned
`KazQADRetrieval` protocol (both the souped checkpoint and off-the-shelf mE5-base, so the
fine-tune delta is measured in the same session).

One-time Kaggle setup (same as the kazeval eval kernel):
1. **Settings → Accelerator**: GPU **T4 x2** (train uses one T4; eval spreads over both).
2. **Settings → Internet**: ON.
3. Token: Kaggle secret `HF_TOKEN` *or* the attached private dataset with `hf_token.txt`
   (account must have accepted [issai/kazparc](https://huggingface.co/datasets/issai/kazparc)
   and [issai/kazqad-retrieval](https://huggingface.co/datasets/issai/kazqad-retrieval)).

The repo is cloned from GitHub `main` — **push before launching**.
Budget: pairs ~15 min (CPU stream) + train ~1-3 h (one T4) + 2 evals ~2 h (T4 x2) —
well inside the 12 h session cap.


In [ ]:
!git clone --depth 1 https://github.com/kekeront/qymyzlm.git /kaggle/working/qymyzlm
%pip install -q /kaggle/working/qymyzlm/embed /kaggle/working/qymyzlm/evallab

import pathlib

from huggingface_hub import login


def hf_token():
    """Kaggle secret HF_TOKEN (interactive runs) or an attached private
    token dataset with hf_token.txt (kernels pushed via the Kaggle API,
    which cannot access UserSecretsClient secrets)."""
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hits = sorted(pathlib.Path("/kaggle/input").glob("**/hf_token.txt"))
        if not hits:
            raise RuntimeError(
                "no HF_TOKEN: add a Kaggle secret or attach the private token dataset"
            ) from None
        return hits[0].read_text().strip()


login(hf_token())


## Stage 1 — KazParC contrastive pairs

Streams `issai/kazparc` (`kazparc_raw`, pinned revision `41df65bd`, gated:auto) and emits
deduped cross-lingual pairs for the default 4 directions `kk-ru,ru-kk,kk-en,en-kk`, e5
prefixes baked in. The full `pairs_kazparc_v0.jsonl` is a **downloadable output artifact**
— grab it from the Output tab so local ablations (kk↔ru vs kk↔en, README §research
question) never need the gated download again. The 10k training subset is cut later by
`train_lim`'s seeded shuffle (seed 42), exactly the pinned protocol.


In [ ]:
# Stage 1 — generate pairs (CPU; streams the gated dataset with the logged-in token).
import subprocess
import sys
from pathlib import Path

PAIRS = Path("/kaggle/working/pairs_kazparc_v0.jsonl")

proc = subprocess.run(
    [
        sys.executable, "-m", "qymyz_embed.data.kazparc_pairs",
        "--output", str(PAIRS),
        # default --directions kk-ru,ru-kk,kk-en,en-kk; default pinned revision
    ],
    text=True,
)
if proc.returncode != 0:
    raise RuntimeError(f"pair generation failed (exit {proc.returncode})")

n_pairs = sum(1 for _ in PAIRS.open(encoding="utf-8"))
print(f"{n_pairs} pairs -> {PAIRS} ({PAIRS.stat().st_size / 1e6:.0f} MB)")
assert n_pairs >= 10_000, f"only {n_pairs} pairs — the 10k Less-is-More cap needs more"


## Stage 2 — Less-is-More fine-tune + model soup

Pinned protocol (all defaults live in `train_lim.py`): 10k-pair seeded cap, FULL
fine-tune, CachedMNRL scale 20, effective batch 512, lr 7e-5, 5 epochs, linear schedule
with 0.2 warmup, fp16 (T4: no bf16). `--mini-batch-size 16` is a **memory/speed knob
only** (GradCache math is invariant to it): the in-code default 8 was reasoned for 8 GB;
Kaggle T4s have 16 GB.

After the soup, the cell **asserts the saved prompt config** carries
`query`/`document`/`passage` prefixes — `document` is the only doc-side prompt key
mteb 2.16.3 honors; without it, eval would encode passages UNPREFIXED and produce a
wrong-but-plausible number (the kazembed-v5 failure mode).


In [ ]:
# Stage 2 — train, then soup. Checkpoint dir names become the record's model name
# (mteb derives it from the path for non-registry models) — name them like model ids.
import json
import subprocess
import sys
from pathlib import Path

CKPT_ROOT = Path("/kaggle/working/checkpoints")
TRAIN_OUT = CKPT_ROOT / "qymyz-embed-lim-v0"
SOUPED = CKPT_ROOT / "qymyz-embed-lim-v0-souped"

for step, argv in [
    (
        "train",
        [
            sys.executable, "-m", "qymyz_embed.train_lim",
            "--data", "/kaggle/working/pairs_kazparc_v0.jsonl",
            "--output", str(TRAIN_OUT),
            "--mini-batch-size", "16",  # 16 GB T4; GradCache-invariant speed knob
        ],
    ),
    (
        "soup",
        [
            sys.executable, "-m", "qymyz_embed.merge",
            str(TRAIN_OUT / "final"), "intfloat/multilingual-e5-base",
            "--output", str(SOUPED),
        ],
    ),
]:
    print(f"===== {step} =====", flush=True)
    proc = subprocess.run(argv, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"{step} failed (exit {proc.returncode})")

# Prompt-config guard: document/passage must both map to "passage: " (see cell above).
prompts = json.loads((SOUPED / "config_sentence_transformers.json").read_text())["prompts"]
expected = {"query": "query: ", "document": "passage: ", "passage": "passage: "}
missing = {k: v for k, v in expected.items() if prompts.get(k) != v}
if missing:
    raise RuntimeError(f"souped checkpoint prompts WRONG (would eval unprefixed): {missing}")
print("prompt config OK:", {k: prompts[k] for k in expected})


## Stage 3 — evaluate on the pinned KazQADRetrieval protocol

Two `kazeval.run_retrieval` subprocesses (per-model isolation, same as the eval kernel):
1. the **souped checkpoint** — mteb's registry-fallback to a bare `SentenceTransformer`
   is **expected and correct** here: the prefixes ride inside the checkpoint's saved
   prompt config, which the Stage 2 guard just verified (unlike the baselines kernel,
   where a fallback means wrong prompts and the record is quarantined);
2. **off-the-shelf `intfloat/multilingual-e5-base`** (registry-resolved) — the same-day,
   same-protocol base row that turns the result into a measured fine-tune delta.

Records land in `/kaggle/working/results/` — download and drop into `evallab/results/`.


In [ ]:
# Stage 3 — eval: souped checkpoint first (the number this kernel exists for).
import subprocess
import sys
from pathlib import Path

RESULTS = Path("/kaggle/working/results")
PER_MODEL_TIMEOUT = 4 * 3600  # mE5-base-sized encoders: ~1 h each on T4 x2 fp16

MODELS = [
    "/kaggle/working/checkpoints/qymyz-embed-lim-v0-souped",
    "intfloat/multilingual-e5-base",
]

for model in MODELS:
    print(f"===== eval {model} =====", flush=True)
    argv = [
        sys.executable, "-m", "kazeval.run_retrieval",
        "--model", model,
        "--tasks", "KazQADRetrieval",
        "--device", "cuda:0,cuda:1",
        "--dtype", "float16",
        "--batch-size", "64",
        "--output-dir", str(RESULTS),
    ]
    try:
        proc = subprocess.run(argv, text=True, timeout=PER_MODEL_TIMEOUT)
    except subprocess.TimeoutExpired:
        print(f"TIMEOUT after {PER_MODEL_TIMEOUT}s: {model}", flush=True)
        continue
    if proc.returncode != 0:
        print(f"FAILED (exit {proc.returncode}): {model}", flush=True)


In [ ]:
# Collect — validate every record (schema-validation rejects half-written files),
# preview the retrieval scores, dump full JSON for download.
from pathlib import Path

from kazeval.results import load_record

RESULTS = Path("/kaggle/working/results")
paths = sorted(RESULTS.glob("*.json"))
print(f"{len(paths)} record file(s) under {RESULTS}\n")

rows, bad = [], []
for p in paths:
    try:
        rows.append(load_record(p))
    except Exception as exc:
        bad.append((p.name, str(exc)))

ret = sorted(
    (r for r in rows if r.task == "KazQADRetrieval"),
    key=lambda r: r.metrics.get("ndcg_at_10", -1.0),
    reverse=True,
)
if ret:
    w = max(len(r.model) for r in ret)
    print(f"{'model':<{w}}  prov      ndcg@10  mrr@10  recall@100")
    for r in ret:
        m = r.metrics
        nan = float("nan")
        print(
            f"{r.model:<{w}}  {r.provenance:<8}  "
            f"{m.get('ndcg_at_10', nan):7.4f}  "
            f"{m.get('mrr_at_10', nan):6.4f}  "
            f"{m.get('recall_at_100', nan):7.4f}"
        )
    print()

for p in paths:
    print(f"----- {p.name} -----")
    print(p.read_text())

if bad:
    print("\n!!! INVALID records (half-written / schema-rejected):")
    for name, err in bad:
        print(f"  {name}: {err}")
